# LGKAT — تسک ۳ و ۵ روی Last.fm (کگل)

این نوت‌بوک فقط **موسیقی (Last.fm)** را با `--att product` و پروتکل همه‌کاربره اجرا می‌کند.
فصل چهارم پایان‌نامه گونه‌ی حاصل‌ضربی را گزارش می‌کند؛ `kg_transformer` را اینجا نزن.

| نشست | `SESSION` | کار | زمان تقریبی |
|---|---:|---|---|
| ۱ | `1` | تسک ۳: رتبه‌بندی همه‌کاربره، ۳ اجرا | حدود ۹ ساعت |
| ۲ | `2` | تسک ۵: سطل‌بندی شروع سرد، ۳ اجرا | حدود ۹ ساعت |

**هر Save Version فقط یک نشست.** اگر هر دو را پشت سر هم بگذاری حدود ۱۸ ساعت می‌شود و سقف ۱۲ ساعته‌ی کگل قطعش می‌کند.

نزن: کتاب، فیلم، رستوران، `--runs 5`، `topk_user_num 0` روی رستوران، بازوی بدون گراف دانش.


## ۰. کارهای دستی قبل از اجرا

1. در کگل یک نوت‌بوک جدید بساز، این فایل را آپلود کن (File → Import Notebook) یا سلول‌ها را کپی کن.
2. سمت راست: **Accelerator = GPU T4 / P100** (۱۶ گیگ).
3. دو دیتاست **خصوصی** بساز و به نوت‌بوک Add Data کن:

**الف) کد — مثلاً `lgkat-code`**  
حداقل این فایل‌ها از مخزن (همان نسخه‌ای که روی سیستم محلی تسک ۱/۲/۴ را داده):

```
src/main.py
src/model.py
src/aggregators.py
src/train_n_runs.py
src/data_loader.py
src/preprocess.py   # اگر در src هست؛ اجباری نیست اگر train صداش نزند
```

می‌توانی کل پوشه‌ی `src/` را زیپ کنی. `data/` را داخل این دیتاست نگذار (حجم و مجوز).

**ب) داده — مثلاً `lgkat-music`**  
از `data/music/` روی سیستم محلی، **حتماً**:

- `ratings_final.npy` (یا `ratings_final.txt`)
- `kg_final.npy` (یا `kg_final.txt`)

اگر موجود است این سه تا را هم بگذار تا ساخت گراف UI تکرار نشود:

- `s_adj_mat.npz`
- `s_norm_adj_mat.npz`
- `s_mean_adj_mat.npz`

فایل‌های `*_final.*` را با `preprocess.py` از نو نساز؛ منفی‌ها عوض می‌شوند و با نتایج قبلی قابل مقایسه نیستند.

4. سلول `SESSION` را روی `1` یا `2` بگذار.
5. **Save Version → Save & Run All** (نه فقط Run داخل ادیتور). با بستن تب، job تا سقف ۱۲ ساعت ادامه دارد.
6. بعد از اتمام: Output را دانلود کن و فایل‌های `result/` را روی سیستم محلی در `LGKAT-Project/result/` کپی کن.


In [ ]:
# 1 = تسک ۳ همه‌کاربره | 2 = تسک ۵ سطل‌بندی
# هر Save Version فقط یکی را بگذار.
SESSION = 1

In [ ]:
import os
import shutil
import sys
from pathlib import Path

os.environ["PYTHONUNBUFFERED"] = "1"
os.environ["PYTHONHASHSEED"] = "0"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

IS_KAGGLE = Path("/kaggle/working").exists()
INPUT = Path("/kaggle/input") if IS_KAGGLE else Path(".")
WORK = Path("/kaggle/working/LGKAT-Project") if IS_KAGGLE else Path(".").resolve()

print("kaggle:", IS_KAGGLE)
print("SESSION:", SESSION)
print("input:", INPUT)
print("work:", WORK)

In [ ]:
def first_match(root, name):
    hits = list(Path(root).rglob(name))
    return hits[0] if hits else None

search_root = INPUT if IS_KAGGLE else WORK
main_py = first_match(search_root, "main.py")
if main_py is None:
    raise FileNotFoundError(
        "main.py پیدا نشد. دیتاست کد را Add Data کن تا src/main.py زیر /kaggle/input باشد."
    )

src_in = main_py.parent
print("src found:", src_in)

ratings = first_match(search_root, "ratings_final.npy") or first_match(search_root, "ratings_final.txt")
kg = first_match(search_root, "kg_final.npy") or first_match(search_root, "kg_final.txt")
if ratings is None or kg is None:
    raise FileNotFoundError(
        "ratings_final یا kg_final پیدا نشد. دیتاست lgkat-music را Add Data کن."
    )

music_in = ratings.parent
print("music found:", music_in)

src_out = WORK / "src"
music_out = WORK / "data" / "music"
result_out = WORK / "result"
src_out.mkdir(parents=True, exist_ok=True)
music_out.mkdir(parents=True, exist_ok=True)
result_out.mkdir(parents=True, exist_ok=True)

for f in src_in.glob("*.py"):
    shutil.copy2(f, src_out / f.name)

needed = [
    "ratings_final.npy", "ratings_final.txt",
    "kg_final.npy", "kg_final.txt",
    "s_adj_mat.npz", "s_norm_adj_mat.npz", "s_mean_adj_mat.npz",
]
for name in needed:
    src_f = music_in / name
    if src_f.exists():
        shutil.copy2(src_f, music_out / name)
        print("copied", name)

assert (music_out / "ratings_final.npy").exists() or (music_out / "ratings_final.txt").exists()
assert (music_out / "kg_final.npy").exists() or (music_out / "kg_final.txt").exists()
print("layout ok")
print("src files:", sorted(p.name for p in src_out.glob("*.py")))
print("music files:", sorted(p.name for p in music_out.iterdir()))

In [ ]:
import tensorflow as tf

print("tf", tf.__version__)
gpus = tf.config.list_physical_devices("GPU")
print("gpus", gpus)
if not gpus:
    raise RuntimeError("GPU دیده نشد. Accelerator را روی GPU بگذار و Save Version را از نو بزن.")

# مدل در TF1 graph mode است؛ eager را همین‌جا هم خاموش کن.
tf.compat.v1.disable_eager_execution()
print("eager disabled")

## فرمان‌های قهرمان Last.fm

پرچم‌ها از `src/cmd_paper_ssl_final.sh` هستند. فقط این‌ها عوض می‌شوند:

- تسک ۳: `--runs 3 --topk_user_num 0 --early_stop_min_epoch 25 --exp_tag protocol_allusers`
- تسک ۵: همان + `--sparsity_eval 1 --sparsity_buckets '[5,10,20]' --exp_tag proposed_sparsity`

خروجی‌ها در `result/` با `exp_tag` از فایل‌های ۱۰۰کاربره جدا می‌مانند.


In [ ]:
MUSIC_CHAMPION = [
    sys.executable, "main.py",
    "--dataset", "music",
    "--aggregator", "concat",
    "--n_epochs", "39",
    "--neighbor_sample_size", "8",
    "--dim", "32",
    "--n_iter", "1",
    "--batch_size", "128",
    "--l2_weight", "1e-4",
    "--lr", "0.005",
    "--layer_size", "[32]",
    "--adj_type", "norm",
    "--alg_type", "lightgcn",
    "--model_type", "KGCN_LightGCN",
    "--node_dropout", "[0.1]",
    "--mess_dropout", "[0.1]",
    "--node_dropout_flag", "1",
    "--alpha", "0.5",
    "--smoothing_steps", "8",
    "--pretrain", "0",
    "--att", "product",
    "--gpu_id", "0",
    "--topk_k_list", "[2,10,50,100]",
    "--topk_user_num", "0",
    "--fusion", "gate",
    "--use_cl", "1",
    "--cl_lambda", "0.02",
    "--cl_tau", "0.2",
    "--use_simgcl", "1",
    "--simgcl_lambda", "0.02",
    "--simgcl_eps", "0.05",
    "--lr_decay", "0.98",
    "--early_stop", "12",
    "--early_stop_min_epoch", "25",
    "--rank_ckpt", "r10",
    "--fast_eval", "1",
    "--log_resources", "1",
    "--runs", "3",
]

def cmd_for_session(session):
    if session == 1:
        return MUSIC_CHAMPION + ["--exp_tag", "protocol_allusers"]
    if session == 2:
        return MUSIC_CHAMPION + [
            "--sparsity_eval", "1",
            "--sparsity_buckets", "[5,10,20]",
            "--exp_tag", "proposed_sparsity",
        ]
    raise ValueError("SESSION must be 1 or 2, got %r" % session)

cmd = cmd_for_session(SESSION)
print(" ".join(cmd))

In [ ]:
import subprocess

src_dir = str(WORK / "src")
print("cwd ->", src_dir)
print("running SESSION", SESSION)

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
env["PYTHONHASHSEED"] = "0"

rc = subprocess.call(cmd, cwd=src_dir, env=env)
print("exit code", rc)
if rc != 0:
    raise SystemExit(rc)

In [ ]:
result_dir = WORK / "result"
print("result files:")
for p in sorted(result_dir.glob("*")):
    print(" ", p.name, p.stat().st_size)

zip_path = Path("/kaggle/working") / ("lgkat_session%s_result.zip" % SESSION) if IS_KAGGLE else WORK / ("lgkat_session%s_result.zip" % SESSION)
if zip_path.exists():
    zip_path.unlink()
shutil.make_archive(str(zip_path.with_suffix("")), "zip", result_dir)
print("zip:", zip_path, zip_path.stat().st_size)

## بعد از دانلود — روی سیستم محلی

زیپ را در `D:\projects\current-git-project\LGKAT-Project\result\` باز کن (فایل‌های قبلی را overwrite نکن مگر `exp_tag` یکی باشد).

از ریشه‌ی مخزن:

```powershell
python scripts\compare_run_to_paper.py result\KGCN_LightGCN_music_product_protocol_allusers.txt music
python scripts\aggregate_final_runs.py --exp_tag protocol_allusers --att product --datasets music --min_runs 3
```

برای تسک ۵ فایل سطل‌ها این است:
`result/KGCN_LightGCN_music_sparsity_product_proposed_sparsity.txt`

نشست ۲: در نوت‌بوک `SESSION = 2` بگذار، Save Version جدید بزن. لازم نیست کد/داده را دوباره آپلود کنی؛ همان Inputها بمانند.
